In [49]:
import torch
import torchvision.models as models
import numpy as np
from transformers import BertTokenizer, BertModel
import torch
from nltk.stem import WordNetLemmatizer, PorterStemmer
import nltk
import torch.nn.functional as F
from torch.optim import Adam
import matplotlib.pyplot as plt
import torch.nn as nn
import json
import datetime
import copy
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
from tqdm import tqdm

# Preprocessing

In [2]:
class VQA:
    def __init__(self, annotation_file=None, question_file=None):
        """
        Constructor of VQA helper class for reading and visualizing questions and answers.
        :param annotation_file (str): location of VQA annotation file
        :return:
        """
        # load dataset
        self.dataset = {}
        self.questions = {}
        self.qa = {}
        self.qqa = {}
        self.imgToQA = {}
        if annotation_file is not None and question_file is not None:
            print('loading VQA annotations and questions into memory...')
            time_t = datetime.datetime.utcnow()
            dataset = json.load(open(annotation_file, 'r'))
            questions = json.load(open(question_file, 'r'))
            print(datetime.datetime.utcnow() - time_t)
            self.dataset = dataset
            self.questions = questions
            self.createIndex()

    def createIndex(self):
        # create index
        print('creating index...')
        imgToQA = {ann['image_id']: [] for ann in self.dataset['annotations']}
        qa = {ann['question_id']: [] for ann in self.dataset['annotations']}
        qqa = {ann['question_id']: [] for ann in self.dataset['annotations']}
        for ann in self.dataset['annotations']:
            imgToQA[ann['image_id']] += [ann]
            qa[ann['question_id']] = ann
        for ques in self.questions['questions']:
            qqa[ques['question_id']] = ques
        print('index created!')

        # create class members
        self.qa = qa
        self.qqa = qqa
        self.imgToQA = imgToQA

    def info(self):
        """
        Print information about the VQA annotation file.
        :return:
        """
        for key, value in self.dataset['info'].items():
            print('%s: %s' % (key, value))

    def getQuesIds(self, imgIds=[], quesTypes=[], ansTypes=[]):
        """
        Get question ids that satisfy given filter conditions. default skips that filter
        :param  imgIds    (int array)   : get question ids for given imgs
                quesTypes (str array)   : get question ids for given question types
                ansTypes  (str array)   : get question ids for given answer types
        :return:    ids   (int array)   : integer array of question ids
        """
        imgIds = imgIds if isinstance(imgIds, list) else [imgIds]
        quesTypes = quesTypes if isinstance(quesTypes, list) else [quesTypes]
        ansTypes = ansTypes if isinstance(ansTypes, list) else [ansTypes]

        if len(imgIds) == len(quesTypes) == len(ansTypes) == 0:
            anns = self.dataset['annotations']
        else:
            if not len(imgIds) == 0:
                anns = sum([self.imgToQA[imgId] for imgId in imgIds if imgId in self.imgToQA], [])
            else:
                anns = self.dataset['annotations']
            anns = anns if len(quesTypes) == 0 else [ann for ann in anns if ann['question_type'] in quesTypes]
            anns = anns if len(ansTypes) == 0 else [ann for ann in anns if ann['answer_type'] in ansTypes]
        ids = [ann['question_id'] for ann in anns]
        return ids

    def getImgIds(self, quesIds=[], quesTypes=[], ansTypes=[]):
        """
        Get image ids that satisfy given filter conditions. default skips that filter
        :param quesIds   (int array)   : get image ids for given question ids
               quesTypes (str array)   : get image ids for given question types
               ansTypes  (str array)   : get image ids for given answer types
        :return: ids     (int array)   : integer array of image ids
        """
        quesIds = quesIds if isinstance(quesIds, list) else [quesIds]
        quesTypes = quesTypes if isinstance(quesTypes, list) else [quesTypes]
        ansTypes = ansTypes if isinstance(ansTypes, list) else [ansTypes]

        if len(quesIds) == len(quesTypes) == len(ansTypes) == 0:
            anns = self.dataset['annotations']
        else:
            if not len(quesIds) == 0:
                anns = sum([self.qa[quesId] for quesId in quesIds if quesId in self.qa], [])
            else:
                anns = self.dataset['annotations']
            anns = anns if len(quesTypes) == 0 else [ann for ann in anns if ann['question_type'] in quesTypes]
            anns = anns if len(ansTypes) == 0 else [ann for ann in anns if ann['answer_type'] in ansTypes]
        ids = [ann['image_id'] for ann in anns]
        return ids

    def loadQA(self, ids=[]):
        """
        Load questions and answers with the specified question ids.
        :param ids (int array)       : integer ids specifying question ids
        :return: qa (object array)   : loaded qa objects
        """
        if isinstance(ids, list):
            return [self.qa[id] for id in ids]
        elif isinstance(ids, int):
            return [self.qa[ids]]

    def showQA(self, anns):
        """
        Display the specified annotations.
        :param anns (array of object): annotations to display
        :return: None
        """
        if len(anns) == 0:
            return 0
        for ann in anns:
            quesId = ann['question_id']
            print("Question: %s" % (self.qqa[quesId]['question']))
            for ans in ann['answers']:
                print("Answer %d: %s" % (ans['answer_id'], ans['answer']))

    def loadRes(self, resFile, quesFile):
        """
        Load result file and return a result object.
        :param   resFile (str)     : file name of result file
        :return: res (obj)         : result api object
        """
        res = VQA()
        res.questions = json.load(open(quesFile))
        res.dataset['info'] = copy.deepcopy(self.questions['info'])
        res.dataset['task_type'] = copy.deepcopy(self.questions['task_type'])
        res.dataset['data_type'] = copy.deepcopy(self.questions['data_type'])
        res.dataset['data_subtype'] = copy.deepcopy(self.questions['data_subtype'])
        res.dataset['license'] = copy.deepcopy(self.questions['license'])

        print('Loading and preparing results...     ')
        time_t = datetime.datetime.utcnow()
        anns = json.load(open(resFile))
        assert isinstance


In [4]:


# Load the JSON file into a DataFrame
annotation_file = "VQA_Dataset/Ans_train/v2_mscoco_train2014_annotations.json"
question_file = "VQA_Dataset/Ques_train/v2_OpenEnded_mscoco_train2014_questions.json"
vqa = VQA(annotation_file, question_file)

# Extract answers and their attributes into arrays
answer_ids = []
question_ids = []
image_ids = []
answers = []

for ann in vqa.dataset['annotations']:
    for ans in ann['answers']:
        answer_ids.append(ans['answer_id'])
        question_ids.append(ann['question_id'])
        image_ids.append(ann['image_id'])
        answers.append(ans['answer'])

# Create a DataFrame from the arrays
answers_df = pd.DataFrame({
    'Answer_ID': answer_ids,
    'Question_ID': question_ids,
    'Image_ID': image_ids,
    'Answer': answers
})

# Print the DataFrame
print(answers_df)


loading VQA annotations and questions into memory...
0:00:16.393023
creating index...
index created!
         Answer_ID  Question_ID  Image_ID   Answer
0                1    458752000    458752      net
1                2    458752000    458752      net
2                3    458752000    458752      net
3                4    458752000    458752  netting
4                5    458752000    458752      net
...            ...          ...       ...      ...
4437565          6    524286005    524286      yes
4437566          7    524286005    524286      yes
4437567          8    524286005    524286      yes
4437568          9    524286005    524286      yes
4437569         10    524286005    524286      yes

[4437570 rows x 4 columns]


In [5]:


# Extract questions and their attributes into arrays
question_ids = []
image_ids = []
questions = []

for ann in vqa.dataset['annotations']:
    question_ids.append(ann['question_id'])
    image_ids.append(ann['image_id'])
    questions.append(vqa.qqa[ann['question_id']]['question'])

# Create a DataFrame from the arrays
questions_df = pd.DataFrame({
    'Question_ID': question_ids,
    'Image_ID': image_ids,
    'Question': questions
})

# Print the DataFrame
print(questions_df)

        Question_ID  Image_ID                                     Question
0         458752000    458752    What is this photo taken looking through?
1         458752001    458752           What position is this man playing?
2         458752002    458752             What color is the players shirt?
3         458752003    458752  Is this man a professional baseball player?
4         262146000    262146                      What color is the snow?
...             ...       ...                                          ...
443752    524286001    524286                  What color is the keyboard?
443753    524286002    524286       Is there a computer mouse on the desk?
443754    524286003    524286                  What color is the computer?
443755    524286004    524286                 Why are there two keyboards?
443756    524286005    524286                            Is that a laptop?

[443757 rows x 3 columns]


In [6]:
# Filter answers_df to keep only the answers with Answer_ID 1
filtered_answers_df = answers_df[answers_df['Answer_ID'] == 1]

# Print the filtered DataFrame
print(filtered_answers_df)


         Answer_ID  Question_ID  Image_ID      Answer
0                1    458752000    458752         net
10               1    458752001    458752     pitcher
20               1    458752002    458752      orange
30               1    458752003    458752         yes
40               1    262146000    262146       white
...            ...          ...       ...         ...
4437520          1    524286001    524286       black
4437530          1    524286002    524286          no
4437540          1    524286003    524286       black
4437550          1    524286004    524286  don't know
4437560          1    524286005    524286         yes

[443757 rows x 4 columns]


In [7]:
print(filtered_answers_df['Answer'])

0                 net
10            pitcher
20             orange
30                yes
40              white
              ...    
4437520         black
4437530            no
4437540         black
4437550    don't know
4437560           yes
Name: Answer, Length: 443757, dtype: object


Using Pytorch

In [8]:

nltk.download('wordnet')

# Initialize WordNet Lemmatizer and Porter Stemmer
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

# Load pre-trained BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

def generate_embeddings(answer, question):
    
    answer_embedding = None
    question_embedding = None

    # Lemmatize and stem answer
    answer = ' '.join([stemmer.stem(lemmatizer.lemmatize(word)) for word in answer.split()])
    
    # Lemmatize and stem question
    question = ' '.join([stemmer.stem(lemmatizer.lemmatize(word)) for word in question.split()])
    
    # Tokenize answer
    tokenized_answer = tokenizer.encode('[CLS] ' + answer + ' [SEP]', add_special_tokens=False, return_tensors='pt')
    print('-------------------------2')
    print('[CLS] ' + answer + ' [SEP]')
    # Tokenize question
    print('-------------------------3')
    print('[CLS] ' + question + ' [SEP]')
    tokenized_question = tokenizer.encode('[CLS] ' + question + ' [SEP]', add_special_tokens=False, return_tensors='pt')

    # Generate embeddings for tokenized answer
    with torch.no_grad():
        answer_output = model(tokenized_answer)
        answer_embedding = answer_output[0].mean(dim=1)  # Use mean pooling over token embeddings
        #embeddings.append(answer_pooled_output)
    
    # Generate embeddings for tokenized question
    with torch.no_grad():
        question_output = model(tokenized_question)
        question_embedding = question_output[0].mean(dim=1)  # Use mean pooling over token embeddings
        #embeddings.append(question_pooled_output)
    
    return answer_embedding, question_embedding

qna_array = []
# code to test the generate_embeddings function
for index, row in filtered_answers_df.iterrows():
    #print(index)
    answer_id = row['Answer_ID']
    question_id = row['Question_ID']
    image_id = row['Image_ID']
    answer_input = row['Answer']
    question_row = questions_df[questions_df['Question_ID'] == question_id]
    question_input = question_row.iloc[0]['Question']
    print('-------------------------1')
    
    print(answer_input, " & ",question_input)
    qna_array.append(question_input)
    qna_array.append(answer_input)

    if (index == 10):
        break
    # # Generate embeddings for answers
    # answer_embeddings = generate_embeddings(answer_input, question_input)
    
    # # Print the embeddings
    # print('-------------------------4')
    # print(answer_embeddings)
    
    # if (index == 0):
    #     break


[nltk_data] Downloading package wordnet to C:\Users\Muhammad
[nltk_data]     Darab\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

C:\Users\Muhammad Darab\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:148: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Muhammad Darab\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

-------------------------1
net  &  What is this photo taken looking through?
-------------------------1
pitcher  &  What position is this man playing?


Using Tensorflow

In [9]:
# from transformers import BertTokenizer, TFBertModel
# import tensorflow as tf
# from nltk.stem import WordNetLemmatizer, PorterStemmer
# import nltk
# nltk.download('wordnet')

# # Initialize WordNet Lemmatizer and Porter Stemmer
# lemmatizer = WordNetLemmatizer()
# stemmer = PorterStemmer()

# # Load pre-trained BERT tokenizer and model
# tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
# model = TFBertModel.from_pretrained('bert-base-uncased')

# def generate_embeddings(answer, question):
#     # Lemmatize and stem answer
#     answer = ' '.join([stemmer.stem(lemmatizer.lemmatize(word)) for word in answer.split()])
    
#     # Lemmatize and stem question
#     question = ' '.join([stemmer.stem(lemmatizer.lemmatize(word)) for word in question.split()])
    
#     # Tokenize answer
#     tokenized_answer = tokenizer.encode_plus('[CLS] ' + answer + ' [SEP]', add_special_tokens=False, return_tensors='tf', padding=True, truncation=True, max_length=512)
    
#     # Tokenize question
#     tokenized_question = tokenizer.encode_plus('[CLS] ' + question + ' [SEP]', add_special_tokens=False, return_tensors='tf', padding=True, truncation=True, max_length=512)

#     # Generate embeddings for tokenized answer
#     answer_output = model(tokenized_answer)
#     answer_embedding = tf.reduce_mean(answer_output[0], axis=1)  # Use mean pooling over token embeddings
    
#     # Generate embeddings for tokenized question
#     question_output = model(tokenized_question)
#     question_embedding = tf.reduce_mean(question_output[0], axis=1)  # Use mean pooling over token embeddings
    
#     return answer_embedding, question_embedding

# for index, row in answers_df.iterrows():
#     answer_id = row['Answer_ID']
#     question_id = row['Question_ID']
#     image_id = row['Image_ID']
#     answer_e = row['Answer']
#     question_row = questions_df[questions_df['Question_ID'] == question_id]
#     question_e = question_row.iloc[0]['Question']

#     print(answer_e, " & ", question_e)

#     # Generate embeddings for answers
#     answer_embeddings, question_embeddings = generate_embeddings(answer_e, question_e)
    
#     # Print the embeddings
#     print(answer_embeddings)
#     print(question_embeddings)
    
#     if index == 0:
#         break


In [15]:
# Define transformations for image preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to 224x224
    transforms.CenterCrop((224, 224)),  # Center crop to 224x224
    transforms.ToTensor(),  # Convert PIL Image to tensor
    transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])  # Normalize image
])

class ImageQADataset(Dataset):
    def __init__(self, images, questions, answers):
        self.images = images
        self.questions = questions
        self.answers = answers
        
    def __getitem__(self, idx):
        image_path = self.images[idx]
        question = self.questions[idx]
        answer = self.answers[idx]

        # Load and preprocess the image
        image_e = Image.open(image_path).convert('RGB')
        image_e.show()
        image_e = self.transform(image)
        answer_e, question_e = generate_embeddings(answer, question)
        return image_e, question_e, answer_e

    def __len__(self):
        return len(self.images)

# Create dataset and dataloader
image_path = ['catt.jpg', 'cattt.jpg']
dataset = ImageQADataset(image_path, [qna_array[0], qna_array[2]], [qna_array[1], qna_array[3]]) # questions_df['Questions'], filtered_answers_df['Answer'])
# print(dataset.answers[1])
# print(dataset.questions[1])
# print(dataset.images[1])
dataloader = DataLoader(dataset, batch_size = 32, shuffle = True)

pitcher
What position is this man playing?
cattt.jpg


# Model

In [3]:
class Encoder(nn.Module):
    def __init__(self, hidden_size, dropout = 0.1):
        super(Encoder, self).__init__()
        self.hidden_size = hidden_size
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first = True)
        self.dropout = nn.Dropout(dropout)
    def forward(self, X):
        Y = self.dropout(X)
        Y, h_s = self.gru(Y)
        return Y, h_s

In [4]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.Wa = nn.Linear(hidden_size, hidden_size)
        self.Ua = nn.Linear(hidden_size, hidden_size)
        self.Va = nn.Linear(hidden_size, 1)
    def forward(self, Q, K):
        attn = self.Va(torch.tanh(self.Wa(Q) + self.Ua(K)))
        attn = attn.squeeze(2).unsqueeze(1)
        weights = F.softmax(attn, dim = -1)
        out = torch.bmm(weights, K)
        return out, weights

In [95]:
class Decoder(nn.Module):
    def __init__(self, hidden_size, output_size, dropout = 0.1, device = 'cpu'):
        super(Decoder, self).__init__()
        self.attention = BahdanauAttention(hidden_size).to(device)
        self.gru = nn.GRU(2 * hidden_size + 512, hidden_size, batch_first = True)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout)
    def forward(self, enc_out, enc_hid, img_emb, target, teacher_force = True):
        dec_hid = enc_hid
        dec_inp = target[:, 0].unsqueeze(1)
        dec_outs = []
        attns = []
        for i in range(1, 16):
            dec_out, dec_hid, attn_weights = self.step(dec_inp, dec_hid, enc_out, img_emb)
            if i != 15:
                if teacher_force:
                    dec_inp = target[:, i].unsqueeze(1)
                else:
                    _, top_i = dec_out.topk(1)
                    dec_inp = top_i.squeeze(-1).detach()
            dec_outs.append(dec_out)
            attns.append(attn_weights)
        dec_outs = torch.cat(dec_outs, dim = 1)
        dec_outs = F.log_softmax(dec_outs, dim = -1)
        attns = torch.cat(attns, dim = 1)
        return dec_outs, dec_hid, attns
    def step(self, X, hidden, enc_out, img_emb):
        out = self.dropout(X)
        Q = hidden.permute(1, 0, 2)
        context, attn_weights = self.attention(Q, enc_out)
        context_with_images = torch.cat((context, img_emb), dim = 2)
        Y = torch.cat((out, context_with_images), dim = 2)
        Y, h_s = self.gru(Y, hidden)
        Y = self.out(Y)
        return Y, h_s, attn_weights

In [69]:
class ImageEncoder(nn.Module):
    def __init__(self, device = 'cpu'):
        super(ImageEncoder, self).__init__()
        self.encoder = models.resnet18(pretrained = True)
        self.encoder = torch.nn.Sequential(*(list(self.encoder.children())[ : -1]))
        self.encoder.eval()
        self.encoder = self.encoder.to(device)
    def forward(self, X):
        with torch.no_grad():
            embedding = self.encoder(X)
        return embedding

In [40]:
def train(dataloader, validation_dataloader, epochs, learning_rate, device, ans_vocab_size, continue_train = False, display = 1, teacher_force = True):
    encoder = Encoder(hidden_size = 300, dropout = 0.1).to(device)
    decoder = Decoder(hidden_size = 300, output_size = ans_vocab_size, dropout = 0.1, device = device).to(device)
    image_encoder = ImageEncoder(device).to(device)
    
    encoder_optimizer = Adam(encoder.params(), lr = learning_rate)
    decoder_optimizer = Adam(decoder.params(), lr = learning_rate)
    
    criterion = nn.NLLLoss()
    
    if continue_train:
        encoder.load_state_dict(torch.load('Encoder.pth'))
        decoder.load_state_dict(torch.load('Decoder.pth'))
        encoder_optimizer.load_state_dict(torch.load('Encoder_Optimizer.pth'))
        decoder_optimizer.load_state_dict(torch.load('Decoder_Optimizer.pth'))
        
    losses = []
    valid_losses = []
    
    for epoch in range(epochs):
        for data in tqdm(dataloader):
            images, encoder_input, decoder_input, target_tensor = data
            images = images.to(device)
            encoder_input = encoder_input.to(device)
            decoder_input = decoder_input.to(device)
            target_tensor = target_tensor.to(device)
            
            encoder_optimizer.zero_grad()
            decoder_optimizer.zero_grad()
            
            images_embeddings = image_encoder(images)
            images_embeddings = images_embeddings.view(len(images_embeddings), -1)
            encoder_output, encoder_hidden = encoder(encoder_input)
            decoder_output, _, _ = decoder(encoder_output, encoder_hidden, images_embeddings[:, np.newaxis, :], decoder_input)
            
            loss = criterion(decoder_output.view(-1, decoder_output.size(-1)), target_tensor.view(-1))
            
            losses += [loss.item()]
            
            loss.backward()
            
            encoder_optimizer.step()
            decoder_optimizer.step()
            
            if (epoch + 1) % display == 0:

                for data in tqdm(validation_dataloader):
                    images, encoder_input, decoder_input, target_tensor = data
                    images = images.to(device)
                    encoder_input = encoder_input.to(device)
                    decoder_input = decoder_input.to(device)
                    target_tensor = target_tensor.to(device)

                    images_embeddings = image_encoder(images)
                    images_embeddings = images_embeddings.view(len(images_embeddings), -1)
                    encoder_output, encoder_hidden = encoder(encoder_input)
                    decoder_output, decoder_hidden, attention_weights = decoder(encoder_output, encoder_hidden, images_embeddings[:, np.newaxis, :], decoder_input, teacher_force)

                    valid_loss = criterion(decoder_output.view(-1, decoder_output.size(-1)), target_tensor.view(-1))
                    valid_losses += [valid_loss.item()]
                    

                print('At Epoch', epoch + 1, 'Current Training Loss:', loss, 'Average Training Loss:', np.average(losses), 'Current Validation Loss', valid_loss, 'Average Validation Loss', np.average(valid_losses))
                step_bins = 20
                length = len(valid_losses)
                num_examples = (length // step_bins) * step_bins
                plt.plot(
                    range(num_examples // step_bins),
                    torch.Tensor(losses[ : num_examples]).view(-1, step_bins). mean(1),
                    label = 'Training Loss'
                )
                plt.plot(
                    range(num_examples // step_bins),
                    torch.Tensor(valid_losses[ : num_examples]).view(-1, step_bins).mean(1),
                    label = 'Validation Loss'
                )
                plt.legend()
                plt.show()
            
        torch.save(encoder.state_dict(), 'Encoder.pth')
        torch.save(decoder.state_dict(), 'Decoder.pth')
        torch.save(encoder_optimizer.state_dict(), 'Encoder_Optimizer.pth')
        torch.save(decoder_optimizer.state_dict(), 'Decoder_Optimizer.pth')

In [ ]:
epochs = 100
learning_rate = 0.001
device = 'cuda'
ans_vocab_size = answers.vocab_size
continue_train = False
teacher_force = True

In [ ]:
train(train_dataloader, validation_dataloader, epochs, learning_rate, device, ans_vocab_size, continue_train, display = 1, teacher_force = teacher_force)